## ⚕️ Task 4: General Health Query Chatbot (Prompt Engineering Based)

### Objective:
Create a chatbot that can answer general health-related questions using an LLM, focusing on prompt engineering for a helpful persona and safety filters.

### 1. API Key Setup

To use the OpenAI API, you'll need an API key. If you don't already have one, create one on the [OpenAI website](https://platform.openai.com/account/api-keys).

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `OPENAI_API_KEY`. Then, we'll retrieve it securely.

In [ ]:
import requests
import re
import textwrap
import time # Import the time module for sleep functionality
from datetime import datetime

# ─────────────────────────────────────────────────────────────────
#   PASTE YOUR OPENROUTER API KEY BELOW (keep the quotes)
# ─────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = "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"

# Free model — no credits needed on OpenRouter
MODEL = "openai/gpt-3.5-turbo"

# Confirm
if "PASTE" in OPENROUTER_API_KEY:
    print("⚠️  Please replace OPENROUTER_API_KEY with your actual key above, then re-run this cell.")
else:
    print("✅ API key set.")
    print(f"   Model : {MODEL}")
    print(f"   Time  : {datetime.now().strftime('%Y-%m-%d %H:%M')}")

✅ API key set.
   Model : openai/gpt-3.5-turbo
   Time  : 2026-05-20 11:09


### 2. Prompt Engineering: Acting as a Helpful Medical Assistant

We will use a `system` message to instruct the LLM to act as a helpful, friendly medical assistant. This guides its tone and persona. We'll also add crucial safety instructions to avoid providing diagnostic or treatment advice.

In [ ]:
def get_medical_assistant_response(user_query):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    messages = [
        {"role": "system", "content": "You are a helpful, friendly, and informative medical assistant. Your goal is to provide general health information and answer common health-related questions. You **must not** provide medical diagnoses, treatment plans, or specific medical advice. Always recommend consulting a qualified healthcare professional for personalized medical guidance. Keep your responses concise and easy to understand."},
        {"role": "user", "content": user_query}
    ]

    data = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": 200,
        "temperature": 0.7
    }

    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=data
        )
        response.raise_for_status() # Raise an exception for HTTP errors
        return response.json()['choices'][0]['message']['content']
    except requests.exceptions.RequestException as e:
        return f"An error occurred with the API request: {e}. Please ensure your API key is correct and try again."
    except KeyError:
        return f"An unexpected response format was received. Response: {response.json()}"

### 3. Safety Filters and Interaction Loop

The most important safety filter for a health chatbot is embedded in the system prompt itself, explicitly telling the LLM not to give medical advice. Additionally, we can add a simple keyword-based filter for highly sensitive queries to reinforce this, although the LLM's own safety mechanisms are usually robust.

Now, let's create a simple loop to interact with our chatbot.

In [ ]:
def run_chatbot():
    print("Hello! I'm your friendly medical assistant. How can I help you today? (Type 'exit' to quit)")
    while True:
        user_input = input("\nYour question: ")
        if user_input.lower() == 'exit':
            print("Thank you for using the medical assistant. Stay healthy!")
            break

        # Simple content filter (can be expanded)
        if any(keyword in user_input.lower() for keyword in ["diagnose", "cure", "treat my", "what should i take for"]):
            print("As a medical assistant, I cannot provide diagnoses or specific treatment recommendations. Please consult a healthcare professional for personalized medical advice.")
            continue

        response = get_medical_assistant_response(user_input)
        print(f"Assistant: {response}")

### 4. Example Queries to Test:

*   "What causes a sore throat?"
*   "Is paracetamol safe for children?"
*   "What are the symptoms of the flu?"
*   "How can I improve my sleep quality?"
*   "I have a cough and fever, what should I do?" (Test safety filter)

In [ ]:
# Run the chatbot
run_chatbot()

Hello! I'm your friendly medical assistant. How can I help you today? (Type 'exit' to quit)
Assistant: Hello! How can I assist you today with any health-related questions or concerns you may have?
Assistant: I recommend consulting a healthcare professional as soon as possible to determine the cause of the pain in that area. It's important to get a proper evaluation and appropriate treatment.
Assistant: A sore throat can be caused by various factors, including viral infections (such as the common cold or flu), bacterial infections (like strep throat), allergies, irritants (such as smoking or air pollution), dry air, or straining your vocal cords. It's important to stay hydrated, get plenty of rest, and consider using lozenges or gargling with warm salt water to relieve symptoms. If your sore throat persists or is severe, it's best to consult a healthcare provider for proper evaluation and guidance.
Assistant: You're welcome! If you have any health-related questions, feel free to ask. I'

KeyboardInterrupt: Interrupted by user